# Ejercicio Unidad 1 — Python para Datos + Limpieza y Normalizacion

## Contexto

Eres el nuevo Data Scientist de una empresa que vende productos de datos (dashboards, reportes, APIs, etc.). El equipo comercial ha mantenido un registro de transacciones en un CSV que fue alimentado por 5 personas distintas, durante 2 anos, sin validacion alguna.

Tu primer encargo: limpiar ese archivo para construir un dashboard de ventas. El gerente quiere ver ingreso por region, por producto y por vendedor. Pero el archivo esta tan sucio que primero necesitas dejarlo usable.

## Reglas

- No borres filas a lo loco. Cada fila es una transaccion real — perder datos es perder plata.
- Documenta cada decision: por que elegiste esa estrategia y no otra.
- Al final, el DataFrame debe tener tipos correctos, sin duplicados, sin valores imposibles, y listo para agrupar.

## Entregable

- Este notebook resuelto.
- Un CSV limpio: `datos_limpios.csv`.
- Un resumen con 3 tablas: ingreso por region, por producto y por vendedor.

---
## Parte 0 — Cargar y explorar

Carga el archivo `datos_feos.csv` y responde estas preguntas antes de tocar nada:

1. Cuantas filas y columnas tiene?
2. Que tipo de dato asigno pandas a cada columna? Alguno esta mal?
3. Cuantos nulos hay por columna?
4. Cuantos duplicados exactos hay?
5. Que valores unicos tiene cada columna de texto? Algo se ve raro?

In [ ]:
import pandas as pd
import numpy as np

# Cargar
df = pd.read_csv('datos_feos.csv')

# Tu codigo aqui


**Escribe aqui tus hallazgos del diagnostico:**

- Filas: ???
- Columnas: ???
- Tipos incorrectos: ???
- Problemas detectados: ???

---
## Parte 1 — Duplicados

Hay filas completamente duplicadas en el dataset.

1. Cuantas filas duplicadas exactas hay?
2. Muestralas para verificar que realmente son duplicados.
3. Eliminalas. Cuantas filas quedan?

**Pista:** usa `.duplicated(keep=False)` para ver todas las ocurrencias, no solo las copias.

In [ ]:
# Tu codigo aqui


---
## Parte 2 — Columna `fecha`

La columna `fecha` es un desastre. Tiene al menos 5 formatos distintos, fechas vacias y basura.

1. Usa `.unique()` para ver todos los formatos que aparecen.
2. Convierte la columna a datetime. `pd.to_datetime()` tiene un parametro `errors='coerce'` que convierte lo que no entiende a NaT (nulo de fecha). Usalo.
3. Cuantas filas quedaron sin fecha (NaT)?
4. Que vas a hacer con esas filas? Justifica.
5. Crea columnas `anio`, `mes` y `dia_semana` a partir de la fecha limpia.

**Pista:** `pd.to_datetime(serie, format='mixed', dayfirst=True)` intenta multiples formatos. Prueba con y sin `dayfirst` y compara resultados — cuidado con fechas como 03-08-2024 que puede ser marzo 8 o agosto 3.

In [ ]:
# Tu codigo aqui


---
## Parte 3 — Columna `producto`

Esta columna tiene varios problemas simultaneos:
- Texto en mayusculas, minusculas y mixto
- Espacios al inicio y final
- Palabras pegadas ('PipelineETL' vs 'Pipeline ETL')
- Valores dentro de corchetes y comillas: `"['Dashboard']"`
- Celdas multi-valor: `"['API REST', 'App Web']"`
- Nulos disfrazados: `None`, `'N/A'`, `'-'`

Tareas:
1. Identifica cuantos valores unicos hay antes de limpiar.
2. Limpia el texto: quitar corchetes y comillas, strip, lowercase.
3. Decide que hacer con los multi-valor (explode o quedarte con el primero).
4. Convierte N/A y - a NaN real.
5. Estandariza los nombres para que queden exactamente 5 productos.
6. Cuantos valores unicos hay despues de limpiar?

**Pista:** `ast.literal_eval()` convierte strings con formato de lista a listas reales de Python. Pero primero verifica que el string empiece con `[`, porque si no, lanza error.

In [ ]:
# Tu codigo aqui


---
## Parte 4 — Columna `region`

Problemas:
- Tildes inconsistentes: 'Bogota' vs 'Bogota'
- Abreviaciones: 'B/quilla', 'Bog'
- Sufijos: 'Bogota D.C.', 'Manizales (Caldas)', 'Medellin (Ant)'
- Case: 'BOGOTA', 'bogota', 'Bogota'
- Vacios disfrazados: espacios, 'Desconocida'

Tareas:
1. Lista todos los valores unicos.
2. Crea un diccionario de mapeo que unifique todas las variantes a 5 ciudades.
3. Aplica el mapeo con `.replace()`.
4. Verifica que solo queden 5 valores unicos (mas NaN si hay).

**Pista:** trabaja en minusculas primero para reducir variantes, luego mapea.

In [ ]:
# Tu codigo aqui


---
## Parte 5 — Columna `vendedor`

El mismo vendedor aparece de muchas formas:
- 'Ana Garcia', 'Garcia, Ana', 'ana.garcia', 'ana.garcia@empresa.com', 'ANA GARCIA'
- 'Sin asignar' y nulos

Tareas:
1. Identifica cuantas formas distintas hay por vendedor.
2. Escribe una funcion `normalizar_vendedor(texto)` que reciba cualquiera de esas formas y retorne 'nombre apellido' en formato titulo.
3. Aplica la funcion a toda la columna.
4. Verifica que queden exactamente 5 vendedores (mas NaN).

**Pista:** piensa en que patron comparten todas las formas. El punto, la coma, el @, las mayusculas cambian, pero las dos palabras clave (nombre y apellido) siempre estan. Puedes usar `.str.contains()` para clasificar o un diccionario de mapeo.

In [ ]:
# Tu codigo aqui


---
## Parte 6 — Columnas `unidades` y `precio_unitario`

Estas columnas deberian ser numericas pero pandas las leyo como `object` (string). Por que?

### unidades
- Tiene texto pegado: '32 unidades', '15uds'
- Valores vacios y 'N/A'
- Valores negativos (imposibles)
- Outliers (valores x100 del rango normal)

### precio_unitario
- Formato moneda: '$1,234.56', '410.98 COP'
- Coma como decimal: '330,31'
- Valores negativos
- Outliers (x100)

Tareas:
1. Limpia ambas columnas para que solo queden numeros.
2. Convierte a float.
3. Reemplaza negativos con NaN.
4. Detecta outliers con IQR. Cuantos hay en cada columna?
5. Decide que hacer con los outliers (eliminar, clip, reemplazar). Justifica.
6. Imputa los nulos con la mediana.
7. Crea la columna `ingreso_total = unidades * precio_unitario`.

**Pista para limpiar texto numerico:** usa `.str.replace()` con regex para quitar todo lo que no sea digito, punto o signo menos. Ejemplo: `serie.str.replace(r'[^\d.\-]', '', regex=True)`

In [ ]:
# Tu codigo aqui


---
## Parte 7 — Columna `calificacion_cliente`

Esta columna es un caos total. Tiene 3 escalas distintas mezcladas:
- Numerica 1-5 (enteros)
- Numerica 1-10 (enteros)
- Texto: 'Bueno', 'Malo', 'Regular', 'Excelente'
- Decimales: 3.5, 2.1
- Vacios y 'N/A'

Tareas:
1. Unifica todo a una escala de 1 a 5.
2. Los valores 1-10 dividelos entre 2 y redondea.
3. Los textos mapealos: Malo=1, Regular=2, Bueno=4, Excelente=5.
4. Los decimales que ya estan en rango 1-5, redondealos.
5. Los que quedan fuera de rango (>5 despues de convertir), pasalos a NaN.

**Pista:** separa el problema en pasos. Primero detecta cuales son texto, cuales son numericos. Trata cada grupo por separado. Luego junta.

In [ ]:
# Tu codigo aqui


---
## Parte 8 — Columna `estado`

Los valores booleanos estan en 10 formatos distintos: 'Activo', 'activo', 'ACTIVO', '1', 'True', 'Si', 'SI', 'Inactivo', '0', 'False', 'No', 'NO', 'Pendiente'.

Tareas:
1. Unifica a una columna booleana: True/False.
2. Que haces con 'Pendiente'? No es ni activo ni inactivo. Decide.
3. Cuantos activos vs inactivos hay?

**Pista:** convierte todo a minusculas primero. Luego usa `.isin()` para clasificar.

In [ ]:
# Tu codigo aqui


---
## Parte 9 — Columna `notas` (Bonus)

La columna de notas tiene texto libre, pero esconde datos utiles:
- Numeros de telefono: 'Contactar al 3101234567'
- Porcentajes de descuento: 'Descuento del 15% aplicado'
- Metodo de pago: 'Pago en efectivo'
- Referencias: 'Ref: FC-1234'

Tareas (todas opcionales, elige las que te interesen):
1. Extrae los numeros de telefono a una columna `telefono`.
2. Extrae el porcentaje de descuento a una columna `descuento`.
3. Extrae el metodo de pago a una columna `metodo_pago`.

**Pista:** expresiones regulares con `str.extract()`. Ejemplo:
```python
df['telefono'] = df['notas'].str.extract(r'(3\d{9})')
```

In [ ]:
# Tu codigo aqui


---
## Parte 10 — Verificacion final y entregable

Antes de guardar, verifica que el DataFrame cumple:

1. `.info()` muestra tipos correctos (datetime, float, category, bool).
2. `.isnull().sum()` — documenta cuantos nulos quedan y por que.
3. `.duplicated().sum()` — debe ser 0.
4. `.describe()` — rangos razonables en unidades, precio e ingreso.
5. Cada columna de texto tiene un numero razonable de valores unicos.

Luego genera las 3 tablas que pidio el gerente:
- Ingreso total por region (ordenado de mayor a menor).
- Ingreso total por producto.
- Ingreso total por vendedor.

Guarda el resultado como `datos_limpios.csv`.

In [ ]:
# Verificacion


In [ ]:
# Tablas para el gerente


In [ ]:
# Guardar
# df.to_csv('datos_limpios.csv', index=False)
